In [ ]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json
import os


In [ ]:
load_dotenv(override=True)
groq_api_key =  os.getenv('GROQ_API_KEY')
openai = OpenAI(api_key=groq_api_key,base_url="https://api.groq.com/openai/v1")

In [ ]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [ ]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
print(summary)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Diya"}
]

In [ ]:
response = openai.chat.completions.create(model="openai/gpt-oss-120b", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Diya"}
]

In [ ]:
response = openai.chat.completions.create(model="openai/gpt-oss-120b", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model="openai/gpt-oss-120b", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Diya"},
    {"role": "assistant", "content": "Well hi there, Diya. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model="openai/gpt-oss-120b", messages=messages)
print(response.choices[0].message.content)

In [ ]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [ ]:
display(Markdown(system_prompt))

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

In [ ]:
response = openai.chat.completions.create(model="openai/gpt-oss-120b", messages=messages)
display(Markdown(response.choices[0].message.content))

In [ ]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [ ]:
record_email_tool("test@testy.com")

In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [ ]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [ ]:
tools

In [ ]:
evaluator_prompt = """
You are an evaluator for a professional digital-twin assistant.

Your job is to determine whether the assistant's response is
STRICTLY related to the person's work, career, education, skills,
projects, experience, or professional background.

Return ONLY one word:

YES
if the response is strictly work/profession related.

NO
if the response contains content unrelated to work, career,
education, skills, projects, experience, or professional background.

Do not judge whether the answer is correct or helpful.
Only judge whether it is professionally relevant.
"""

In [ ]:
#exercise mking the evaluator function to answer only work related queires
def evaluate_response(response_text):
    evaluation_messages = [
        {"role": "system", "content": evaluator_prompt},
        {
            "role": "user",
            "content": f"Evaluate this assistant response:\n\n{response_text}"
        }
    ]

    evaluation = openai.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=evaluation_messages
    )

    return evaluation.choices[0].message.content.strip()

In [ ]:
def chat(message, history):

    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )

    # LLM CALL 1
    response = openai.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        tools=tools
    )

    # AGENT LOOP
    while response.choices[0].finish_reason == "tool_calls":

        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        for tool_call in assistant_message.tool_calls:

            if tool_call.function.name == "record_email_tool":

                arguments = json.loads(
                    tool_call.function.arguments
                )

                email = arguments["email"]

                result = record_email_tool(email)

                messages.append({
                    "role": "tool",
                    "content": result,
                    "tool_call_id": tool_call.id
                })

        response = openai.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=messages,
            tools=tools
        )

    # FINAL ANSWER
    answer = response.choices[0].message.content

    # LLM CALL 2 — EVALUATOR
    evaluation = evaluate_response(answer)

    print("Evaluator:", evaluation)

    # RETURN BASED ON EVALUATION
    if evaluation.strip().upper() == "YES":
        return answer

    else:
        return (
            "I'm here to help with questions related to my "
            "professional background, skills, projects, education, "
            "and experience."
        )

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

In [ ]:
#An ai assistant for buisnesses

business_system_prompt = """You are the digital twin of Micheal James the CEO of company named GVM.
It is a service based company, operating in Hyderabad, catering to major clients across the world.
Answer questions asked by clients and record their emails if they show interest to communicate to the real person."""


In [ ]:
def chat(message, history):

    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = (
        [{"role": "system", "content": business_system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )

    # LLM CALL 1
    response = openai.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        tools=tools
    )

    # AGENT LOOP
    while response.choices[0].finish_reason == "tool_calls":

        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        for tool_call in assistant_message.tool_calls:

            if tool_call.function.name == "record_email_tool":

                arguments = json.loads(
                    tool_call.function.arguments
                )

                email = arguments["email"]

                result = record_email_tool(email)

                messages.append({
                    "role": "tool",
                    "content": result,
                    "tool_call_id": tool_call.id
                })

        response = openai.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=messages,
            tools=tools
        )

    # FINAL ANSWER
    answer = response.choices[0].message.content

    
    return answer

 

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)